# Hunting Hidden Galaxy Collisions with AI
Welcome to the interactive demonstration for our galaxy merger and anomaly detection pipeline! 

This notebook will quickly walk you through downloading a small sample of SDSS data, processing it with a ResNet50 deep learning model, detecting structural anomalies, and scanning for multi-component systems (like mergers) directly in the image plane.

### Step 1: Environment Setup
**If you are running this in Google Colab**, uncomment the code below to clone the repository, install dependencies, and navigate into the project folder.

In [ ]:
# !git clone https://github.com/JohnCassavetes/astro1.git
# %cd astro1
# !pip install -r requirements.txt

### Step 2: Configure for a Fast Demo
By default, the pipeline scales to thousands of images. For this quick 2-minute demo, we will programmatically override the configuration to download only **100** SDSS galaxies instead of 5,000.

In [ ]:
import yaml

DEMO_CONFIG = "config.demo.yaml"

with open(DEMO_CONFIG, "r") as f:
    config = yaml.safe_load(f)

config['pipeline']['sdss']['limit'] = 100

with open(DEMO_CONFIG, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(f"Demo configuration updated successfully: {DEMO_CONFIG}")

### Step 3: Run the Pipeline Modules
Now we will sequentially run the 5 core modules of the pipeline directly from the command line interface (CLI). Watch the output logs below to understand what each stage is doing!

In [ ]:
import os
import subprocess
import sys
import yaml

DEMO_CONFIG = "config.demo.yaml"

with open(DEMO_CONFIG, "r") as f:
    config = yaml.safe_load(f)

demo_limit = int(config["pipeline"]["sdss"].get("limit", 100))
demo_env = os.environ.copy()
demo_env["ASTRO1_CONFIG"] = DEMO_CONFIG
demo_env["PYTHONUNBUFFERED"] = "1"

def run_stage(title, cmd):
    print(title, flush=True)
    display_cmd = ['python', '-u', *cmd[1:]] if cmd and cmd[0] == 'python' else cmd
    print(f"$ ASTRO1_CONFIG={DEMO_CONFIG} {' '.join(display_cmd)}", flush=True)
    if cmd and cmd[0] == 'python':
        cmd = ['python', '-u', *cmd[1:]]
    process = subprocess.Popen(
        cmd,
        env=demo_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    while True:
        chunk = process.stdout.read(1024)
        if not chunk:
            break
        sys.stdout.write(chunk.decode('utf-8', errors='replace'))
        sys.stdout.flush()
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"{title} failed with exit code {return_code}")

run_stage(
    f"--- STAGE 1: Downloading {demo_limit} SDSS cutouts ---",
    ["python", "scripts/download_data.py", "--n", str(demo_limit)],
)
run_stage("\n--- STAGE 2: Preprocessing and Normalizing ---", ["python", "scripts/preprocess_images.py"])
run_stage("\n--- STAGE 3: Extracting ResNet50 Embeddings ---", ["python", "scripts/generate_embeddings.py"])
run_stage("\n--- STAGE 4: Isolation Forest Anomaly Detection ---", ["python", "scripts/detect_anomalies.py"])
run_stage("\n--- STAGE 5: Scanning Raw Images for Mergers ---", ["python", "scripts/scan_raw_secondary_sources.py"])


### Step 4: Visualize the Results
The scanner script (Stage 5) automatically drew red bounding boxes around the primary galaxies and cyan bounding boxes around any high-confidence secondary/multi-component galaxies it found physically nearby.

Let's visualize any successful candidates!

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image

# Find all generated overlay diagnostic images
overlay_dir = "demo/results/final/raw_object_scan/overlays"
overlays = glob.glob(os.path.join(overlay_dir, "*.png"))

if not overlays:
    print("Out of 100 galaxies, no strong merger candidates were found in this batch!")
    print("Try increasing the limit in config.demo.yaml to 500 and running again.")
else:
    print(f"Found {len(overlays)} strong candidates! Plotting them below:")
    
    # Plot up to 4 images max
    num_to_plot = min(len(overlays), 4)
    fig, axes = plt.subplots(1, num_to_plot, figsize=(15, 5))
    
    # Handle single image case
    if num_to_plot == 1:
        axes = [axes]
        
    for ax, img_path in zip(axes, overlays[:num_to_plot]):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis('off')
        # Set title to objid
        ax.set_title(os.path.basename(img_path).split('_')[0])
        
    plt.tight_layout()
    plt.show()